In [1]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. CONFIGURAÇÕES INICIAIS
# ==========================================
np.random.seed(42)
dias_simulacao = 90 # 3 meses de dados
frequencia_minutos = 1

print("Iniciando a geração de dados do Smart Bus (Versão 3 - Física Corrigida)...")

# Gerando o eixo de tempo (3 meses, rodando 19h por dia, das 05h às 23h59)
datas = pd.date_range(start="2026-03-01 05:00:00", periods=(dias_simulacao * 19 * 60), freq=f'{frequencia_minutos}min')
df = pd.DataFrame({'data_hora': datas})

# ==========================================
# 2. FEATURES TEMPORAIS E DE CONTEXTO
# ==========================================
df['hora'] = df['data_hora'].dt.hour
df['dia_semana'] = df['data_hora'].dt.dayofweek
df['is_horario_pico'] = np.where(
    ((df['hora'] >= 6) & (df['hora'] <= 8)) | ((df['hora'] >= 17) & (df['hora'] <= 19)), 1, 0
)

# ==========================================
# 3. FEATURES AMBIENTAIS E OPERACIONAIS
# ==========================================
# Simulando Temperatura Externa (Curva senoidal ao longo do dia)
variacao_diaria = np.sin((df['hora'] - 9) * (np.pi / 12)) 
df['temp_externa'] = 25 + (variacao_diaria * 7) + np.random.normal(0, 0.5, len(df))

# Simulando Incidência Solar (Pico ao meio-dia)
df['incidencia_solar'] = np.clip(np.sin((df['hora'] - 6) * (np.pi / 12)) * 100, 0, 100)
df['incidencia_solar'] += np.random.normal(0, 5, len(df))
df['incidencia_solar'] = np.clip(df['incidencia_solar'], 0, 100)

# Simulando Lotação (Mais cheio nos horários de pico e dias úteis)
fator_fds = np.where(df['dia_semana'] >= 5, 0.5, 1.0)
base_lotacao = np.where(df['is_horario_pico'] == 1, np.random.randint(50, 90, len(df)), np.random.randint(10, 40, len(df)))
df['lotacao'] = (base_lotacao * fator_fds).astype(int)

# Simulando Dinâmica de Paradas e Portas (15% do tempo no ponto com porta aberta)
df['portas_abertas'] = np.random.choice([0, 1], size=len(df), p=[0.85, 0.15])
df['velocidade_kmh'] = np.where(df['portas_abertas'] == 1, 0, np.random.uniform(20, 60, len(df)))

# Simulando Carga Térmica e Temperatura Interna (A física do calor)
carga_termica = (df['temp_externa'] - 22) * 0.5 + (df['lotacao'] * 0.1) + (df['incidencia_solar'] * 0.05) + (df['portas_abertas'] * 5)
df['temp_interna_atual'] = 22 + carga_termica + np.random.normal(0, 0.5, len(df))

# ==========================================
# 4. TARGETS E COMPORTAMENTO NORMAL (NERFADO)
# ==========================================
# Target 1: A Potência ideal que o modelo de Regressão vai prever
potencia_necessaria = carga_termica * 12 
df['potencia_hvac_target'] = np.clip(potencia_necessaria, 0, 100)

# Consumo de energia padrão do equipamento saudável
# Ruído : Desvio padrão de 1.5 para simular flutuação da rede elétrica
df['consumo_kw'] = (df['potencia_hvac_target'] / 100) * 18.0 + np.random.normal(0, 1.5, len(df))

# ==========================================
# 5. INJEÇÃO DE ANOMALIAS (FÍSICA CORRIGIDA: DIAS INTEIROS)
# ==========================================
# Coluna de gabarito zerada
df['is_falha_mecanica'] = 0

# Apenas a data (sem a hora) para sortear dias inteiros
df['data_apenas'] = df['data_hora'].dt.date
dias_unicos = df['data_apenas'].unique()

# Sortear 3 dias aleatórios em que o ônibus rodou com defeito
dias_com_defeito = np.random.choice(dias_unicos, size=3, replace=False)

# Criar uma máscara para selecionar todas as linhas (minutos) desses 3 dias
mascara_defeito = df['data_apenas'].isin(dias_com_defeito)

# Marcar o gabarito (Ground Truth) como 1
df.loc[mascara_defeito, 'is_falha_mecanica'] = 1

# Injetar o consumo extra persistente durante todo o dia!
# Ex: Um filtro muito sujo puxando em média +3.5kW de forma contínua
df.loc[mascara_defeito, 'consumo_kw'] += np.random.normal(3.5, 0.5, mascara_defeito.sum())

# Limpar a coluna auxiliar de data
df.drop(columns=['data_apenas'], inplace=True)

# ==========================================
# 6. EXPORTAÇÃO
# ==========================================
# Cria a pasta 'data' no diretório atual caso ela não exista
os.makedirs('data', exist_ok=True)

# Atualiza o nome do arquivo para incluir o caminho da pasta
nome_arquivo = 'data/dataset_smart_bus_v3.csv'
df.to_csv(nome_arquivo, index=False)

print(f"\nConcluído! Arquivo '{nome_arquivo}' gerado com sucesso.")
print(f"Total de registros: {len(df)} linhas.")
print(f"Dias em que o ônibus rodou com falha mecânica: {dias_com_defeito}")

Iniciando a geração de dados do Smart Bus (Versão 3 - Física Corrigida)...

Concluído! Arquivo 'data/dataset_smart_bus_v3.csv' gerado com sucesso.
Total de registros: 102600 linhas.
Dias em que o ônibus rodou com falha mecânica: [datetime.date(2026, 3, 15) datetime.date(2026, 4, 2)
 datetime.date(2026, 3, 18)]


In [2]:
print(df.head())

            data_hora  hora  dia_semana  is_horario_pico  temp_externa  \
0 2026-03-01 05:00:00     5           6                0     19.186179   
1 2026-03-01 05:01:00     5           6                0     18.868690   
2 2026-03-01 05:02:00     5           6                0     19.261666   
3 2026-03-01 05:03:00     5           6                0     19.699337   
4 2026-03-01 05:04:00     5           6                0     18.820745   

   incidencia_solar  lotacao  portas_abertas  velocidade_kmh  \
0          0.000000       19               0       59.679970   
1          6.847881       15               1        0.000000   
2          0.000000        7               1        0.000000   
3          0.000000        5               0       58.869554   
4          0.000000        8               0       45.362064   

   temp_interna_atual  potencia_hvac_target  consumo_kw  is_falha_mecanica  
0           23.125544              5.917076    2.165897                  0  
1           28.3

In [4]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score,
    classification_report, 
    confusion_matrix
)

print("="*50)
print("INICIANDO TREINAMENTO UNIFICADO - SMART BUS V3")
print("="*50)

# ==========================================
# 1. Carregamento e Preparação dos Dados
# ==========================================
print("\n[1/5] A carregar dataset e a ordenar no tempo...")
caminho_dados = 'data/dataset_smart_bus_v3.csv'
df = pd.read_csv(caminho_dados)

# Ordenação cronológica fundamental para a média móvel funcionar corretamente
df = df.sort_values(by='data_hora').reset_index(drop=True)

# ==========================================
# 2. Motor de Otimização (Regressão HVAC)
# ==========================================
print("\n[2/5] A treinar Motor de Otimização (HVAC)...")

features_base = [
    'hora', 'dia_semana', 'is_horario_pico', 'temp_externa', 
    'incidencia_solar', 'lotacao', 'portas_abertas', 
    'velocidade_kmh', 'temp_interna_atual'
]

X_reg = df[features_base]
y_reg = df['potencia_hvac_target']

# Split temporal (sem shuffle) para não prever o passado com dados do futuro
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42, shuffle=False
)

# HistGradientBoostingRegressor com PODA (Pruning)
regressor_gb = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.1,
    max_depth=7,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    random_state=42
)
regressor_gb.fit(X_reg_train, y_reg_train)

y_reg_pred = regressor_gb.predict(X_reg_test)

rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
mae = mean_absolute_error(y_reg_test, y_reg_pred)
r2 = r2_score(y_reg_test, y_reg_pred)

print(f"-> RMSE (Erro Quadrático Médio): {rmse:.2f} kW")
print(f"-> MAE (Erro Absoluto Médio): {mae:.2f} kW")
print(f"-> R² (Coeficiente de Determinação): {r2:.4f} ({(r2*100):.1f}% da variância explicada)")

# ==========================================
# 3. Engenharia de Features (Resíduos)
# ==========================================
print("\n[3/5] A calcular features temporais (Resíduo e Média Móvel)...")

# Usa o modelo recém-treinado para descobrir qual deveria ser o consumo em todo o dataset
df['potencia_esperada'] = regressor_gb.predict(X_reg)

# Consumo esperado (transformando a potência de % ou kW para a escala do autocarro)
df['consumo_esperado'] = (df['potencia_esperada'] / 100) * 18.0 
df['residuo_consumo'] = df['consumo_kw'] - df['consumo_esperado']

# A "Lembrança" dos últimos 15 minutos para anular o ruído elétrico
df['residuo_media_15m'] = df['residuo_consumo'].rolling(window=15, min_periods=1).mean()

# ==========================================
# 4. Motor de Manutenção Preditiva (Classificação)
# ==========================================
print("\n[4/5] A treinar Motor de Manutenção (Deteção de Falhas)...")

features_clf = ['potencia_hvac_target', 'consumo_kw', 'residuo_consumo', 'residuo_media_15m']
target_clf = 'is_falha_mecanica'

X_clf = df[features_clf]
y_clf = df[target_clf]

# Usando stratify para garantir que falhas e normalidade sejam distribuídas igualmente
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# HistGradientBoostingClassifier com PODA (Pruning)
clf_gb = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.1,
    max_depth=5,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    class_weight='balanced', 
    random_state=42
)
clf_gb.fit(X_clf_train, y_clf_train)

# ==========================================
# 5. Avaliação de Negócio (Threshold Tuning)
# ==========================================
print("\n[5/5] A avaliar Regras de Negócio e a Guardar Modelos...")

probabilidades = clf_gb.predict_proba(X_clf_test)[:, 1]

# Limiar de 25% definido na regra de negócio
LIMIAR_NEGOCIO = 0.25 
y_pred_custom = (probabilidades >= LIMIAR_NEGOCIO).astype(int)

print(f"\nResultados com Limiar de Decisão em {LIMIAR_NEGOCIO * 100}%:")
print(classification_report(y_clf_test, y_pred_custom, target_names=["Normal (0)", "Falha (1)"]))

cm_clf = confusion_matrix(y_clf_test, y_pred_custom)
print("Matriz de Confusão:")
print(f"Verdadeiro Normal: {cm_clf[0][0]} | Falso Positivo (Mecânico foi à toa): {cm_clf[0][1]}")
print(f"Falso Negativo:    {cm_clf[1][0]}   | Verdadeiro Falha (Autocarro salvo):      {cm_clf[1][1]}")

# ==========================================
# 6. Exportação Otimizada em Pastas
# ==========================================
# Criar pasta 'models' se não existir
os.makedirs('models', exist_ok=True)

caminho_regressor = 'models/hvac_regressor_gb.pkl'
caminho_classificador = 'models/modelo_manutencao_gradient_boosting.pkl'

joblib.dump(regressor_gb, caminho_regressor)
joblib.dump(clf_gb, caminho_classificador)

print(f"\nSucesso! Modelos guardados na pasta 'models':")
print(f"- {caminho_regressor}")
print(f"- {caminho_classificador}")

INICIANDO TREINAMENTO UNIFICADO - SMART BUS V3

[1/5] A carregar dataset e a ordenar no tempo...

[2/5] A treinar Motor de Otimização (HVAC)...
-> RMSE (Erro Quadrático Médio): 1.38 kW
-> MAE (Erro Absoluto Médio): 0.72 kW
-> R² (Coeficiente de Determinação): 0.9987 (99.9% da variância explicada)

[3/5] A calcular features temporais (Resíduo e Média Móvel)...

[4/5] A treinar Motor de Manutenção (Deteção de Falhas)...

[5/5] A avaliar Regras de Negócio e a Guardar Modelos...

Resultados com Limiar de Decisão em 25.0%:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     19656
   Falha (1)       0.97      0.99      0.98       864

    accuracy                           1.00     20520
   macro avg       0.98      1.00      0.99     20520
weighted avg       1.00      1.00      1.00     20520

Matriz de Confusão:
Verdadeiro Normal: 19629 | Falso Positivo (Mecânico foi à toa): 27
Falso Negativo:    5   | Verdadeiro Falha (Autocarro salvo):  

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os # Adicionado para manipulação de pastas

# Cria a pasta 'images' caso ela não exista
os.makedirs('images', exist_ok=True)

print("Carregando dados para visualização...")
# Atualizado para ler o dataset da pasta 'data'
df = pd.read_csv('data/dataset_smart_bus_v3.csv')

# Garantindo que o tempo seja tratado corretamente
df['data_hora'] = pd.to_datetime(df['data_hora'])
df = df.sort_values(by='data_hora').reset_index(drop=True)

# ---------------------------------------------------------
# 1. Recriando as features necessárias para o gráfico
# ---------------------------------------------------------
# Assumindo que a potência alvo já dita o consumo esperado:
df['consumo_esperado'] = (df['potencia_hvac_target'] / 100) * 18.0 
df['residuo_consumo'] = df['consumo_kw'] - df['consumo_esperado']
df['residuo_media_15m'] = df['residuo_consumo'].rolling(window=15, min_periods=1).mean()

# ---------------------------------------------------------
# 2. Configuração Visual Geral (Estilo Profissional)
# ---------------------------------------------------------
sns.set_theme(style="darkgrid")
plt.rcParams.update({'font.size': 12})

# =========================================================
# GRÁFICO 1: A "Física" do Smart Bus (Dispersão)
# =========================================================
print("Gerando Gráfico 1: Comportamento do HVAC...")
plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(
    data=df.sample(5000, random_state=42), # Amostra para o gráfico não ficar uma mancha sólida
    x='temp_externa', 
    y='potencia_hvac_target', 
    hue='lotacao', 
    palette='flare', 
    alpha=0.7,
    edgecolor=None
)
plt.title('Comportamento do HVAC: Temperatura Externa vs. Potência Alvo\n(Colorido por Lotação do Ônibus)', fontsize=14, weight='bold')
plt.xlabel('Temperatura Externa (°C)')
plt.ylabel('Potência Alvo do Ar Condicionado (%)')
plt.legend(title='Lotação (Passageiros)')
plt.tight_layout()

# Atualizado para salvar na pasta 'images'
plt.savefig('images/grafico_1_hvac_fisica.png', dpi=300)
plt.close()

# =========================================================
# PREPARAÇÃO PARA OS GRÁFICOS 2 E 3 (Zoom em uma Falha)
# =========================================================
# Ver 3 meses de dados em uma linha só vira um borrão. 
# Vamos isolar uma janela de 3 dias ao redor da PRIMEIRA falha mecânica.
indice_primeira_falha = df[df['is_falha_mecanica'] == 1].index[0]

# Pegamos 1 dia (1440 minutos) antes da falha e 2 dias depois dela
janela_inicio = max(0, indice_primeira_falha - 1440)
janela_fim = min(len(df), indice_primeira_falha + 2880)
df_zoom = df.iloc[janela_inicio:janela_fim]

# =========================================================
# GRÁFICO 2 e 3: O Problema e a Solução Lado a Lado
# =========================================================
print("Gerando Gráficos 2 e 3: O poder da Média Móvel...")
fig, eixos = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Cores para indicar o status real do ônibus no fundo do gráfico
cores_fundo = df_zoom['is_falha_mecanica'].map({0: 'white', 1: '#ffcccc'})

# --- Plot Superior: O Problema (Resíduo Cru) ---
eixos[0].plot(df_zoom['data_hora'], df_zoom['residuo_consumo'], color='gray', alpha=0.8, label='Resíduo Instantâneo (Ruído)')
eixos[0].set_title('O Problema: Ruído Elétrico Ocultando a Falha Mecânica', fontsize=14, weight='bold')
eixos[0].set_ylabel('Variação de Consumo (kW)')
eixos[0].axhline(0, color='black', linestyle='--', linewidth=1)

# Pintar o fundo onde há falha
eixos[0].fill_between(df_zoom['data_hora'], eixos[0].get_ylim()[0], eixos[0].get_ylim()[1], 
                      where=(df_zoom['is_falha_mecanica'] == 1), color='red', alpha=0.15, label='Período de Falha Real')
eixos[0].legend(loc='upper left')

# --- Plot Inferior: A Solução (Média Móvel 15m) ---
eixos[1].plot(df_zoom['data_hora'], df_zoom['residuo_media_15m'], color='#2ca02c', linewidth=2, label='Média Móvel 15m (Sinal Limpo)')
eixos[1].set_title('A Solução: Feature Temporal Revelando a Anomalia Contínua', fontsize=14, weight='bold')
eixos[1].set_ylabel('Variação de Consumo (kW)')
eixos[1].set_xlabel('Data e Hora')
eixos[1].axhline(0, color='black', linestyle='--', linewidth=1)

# Pintar o fundo onde há falha
eixos[1].fill_between(df_zoom['data_hora'], eixos[1].get_ylim()[0], eixos[1].get_ylim()[1], 
                      where=(df_zoom['is_falha_mecanica'] == 1), color='red', alpha=0.15, label='Período de Falha Real')

# Adicionar a linha do Limiar de Negócio
eixos[1].axhline(1.5, color='darkred', linestyle=':', linewidth=2, label='Limiar de Alerta do Modelo')
eixos[1].legend(loc='upper left')

plt.tight_layout()

# Atualizado para salvar na pasta 'images'
plt.savefig('images/grafico_2_3_ruido_vs_sinal.png', dpi=300)
plt.close()

print("\nImagens geradas com sucesso na pasta 'images'!")
print("- images/grafico_1_hvac_fisica.png")
print("- images/grafico_2_3_ruido_vs_sinal.png")

Carregando dados para visualização...
Gerando Gráfico 1: Comportamento do HVAC...
Gerando Gráficos 2 e 3: O poder da Média Móvel...

Imagens geradas com sucesso na pasta 'images'!
- images/grafico_1_hvac_fisica.png
- images/grafico_2_3_ruido_vs_sinal.png
